In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_22.pickle

trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  18
me:  26
me:  6
me:  8
me:  10
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  18
me:  26
me:  6
me:  8
me:  10
trying: ['question_name', 'question_of_interest_cell18']
me:  6
me:  6
trying: ['question_name', 'question_of_interest_cell18']
me:  6
me:  6
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  18
me:  26
me:  6
me:  8
me:  10
trying: ['title_for_x_axis_cell29', 'title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  18
me:  26
me:  6
me:  8
me:  10
trying: ['

In [4]:
%%RecordEvent
%%time
### cell 23 ###

### cell 23 optimized ###

def grab_subset_of_data_35(original_df, question_of_interest):
    # select and rename only the response columns
    subset_cols = [c for c in original_df.columns if question_of_interest in c]
    mapping = {c: c.split('-')[-1].lstrip() for c in subset_cols}
    df = original_df[subset_cols].rename(columns=mapping)
    # drop rows where all response columns are null, using GPU‐accelerated isnull/all
    df = df.loc[~df.isnull().all(axis=1)]
    return df


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(question_of_interest,
                                                                                                    include_2017=False):
    # collect (df_source, year) pairs
    sources = [
        (responses_df_2018_cell10, '2018'),
        (responses_df_2019_cell10, '2019'),
        (responses_df_2020,      '2020'),
        (responses_df_2021,      '2021'),
        (responses_df_2022_cell10,'2022'),
    ]
    if include_2017:
        sources.insert(0, (responses_df_2017, '2017'))
    # build list of cleaned slices
    df_list = [grab_subset_of_data_35(df_src, question_of_interest) for df_src, _ in sources]
    years   = [yr for _, yr in sources]
    # concat with keys->year index, then reset_index to get a 'year' column
    df_combined = pd.concat(df_list, keys=years, names=['year']).reset_index(level='year')
    # count non-null responses per column per year
    df_counts   = df_combined.groupby('year').count().reset_index()
    return df_combined, df_counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # compute total respondents per year
    totals = df.groupby('year').size().sort_index()
    # vectorized GPU division and multiplication
    df_pct = (
        df_counts
        .set_index('year')
        .astype('float')
        .div(totals, axis=0)
        * 100
    )
    return df_pct.reset_index()

# ---- usage ----
question = 'What programming languages do you use on a regular basis?'
language_df_combined, language_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question
    )
)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(
    language_df_combined,
    language_df_combined_counts
)
# select and reshape for plotting
selected_cols = ['year','Python','SQL','C++','C','R','Java','Javascript','Other']
language_df_combined_percentages_cell35 = language_df_combined_percentages[selected_cols]
df_cell35 = (
    language_df_combined_percentages_cell35
    .melt(id_vars=['year'], value_vars=selected_cols)
    .sort_values(by=['year','value'], ascending=True)
    .rename(columns={'variable': ''})
)
df_cell35.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 40 entries, 10 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.3+ KB
CPU times: user 839 ms, sys: 39.3 ms, total: 879 ms
Wall time: 887 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_23_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_23_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[23], f)


In [ ]:
opt_output = Out.get(4)